<a href="https://colab.research.google.com/github/5556mani/AI-Engineering/blob/main/PyTorch/9_optimising_ANN_using_optunia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# import libraries
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as opt

In [2]:

device=('cuda' if torch.cuda.is_available() else 'cpu')
device

'cpu'

In [4]:

df=pd.read_csv("fmnist_small.csv")

In [5]:

# setting seed
torch.manual_seed(42)

In [6]:

# splitting the dataset in train and test
X=df.iloc[:,1:].values
Y=df.iloc[:,0].values
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

X_test=X_test/255.0
X_train=X_train/255.0


In [9]:
class custom_DATA_class(Dataset):

  def __init__(self,features,labels):
    self.features=torch.tensor(features, dtype=torch.float32)
    self.labels=torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]




In [10]:
# creting dataset
train_dataset=custom_DATA_class(X_train,Y_train)
test_dataset=custom_DATA_class(X_test,Y_train)


In [11]:
# creating dataloader
test_laoder=DataLoader(test_dataset,batch_size=32,shuffle=True)
train_laoder=DataLoader(train_dataset,batch_size=32,shuffle=False)

In [23]:
# define NN class
class NN(nn.Module):

  def __init__(self,inputshape,outputshape,num_layer,neuro_in_layer,p_D):

    super().__init__()

    layer=[]

    for i in range(num_layer):
      layer.append(nn.Linear(inputshape,neuro_in_layer))  # we can try to optimise such that we can vary no of neurons in each layer
      layer.append(nn.BatchNorm1d(neuro_in_layer))
      layer.append(nn.ReLU())
      layer.append(nn.Dropout(p_D))
      inputshape=neuro_in_layer

    layer.append(nn.Linear(neuro_in_layer, outputshape))

    self.model = nn.Sequential(*layer)

  def forward(self,x):
    return self.model(x)


In [24]:
# objective function
def objective(trial):

  # next hyperparameter values from the search space
  num_layer = trial.suggest_int("num_hidden_layers", 1, 5)
  neuro_in_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)  # we can try to optimise such that we can vary no of neurons in each layer , for that we need this to be a list
  epochs = trial.suggest_int("epochs", 10, 50, step=10)
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  p_D = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

  # model init
  inputshape = 784
  outputshape = 10

  model = NN(inputshape,outputshape,num_layer,neuro_in_layer,p_D)
  model.to(device)

  # optimizer selection
  criterion = nn.CrossEntropyLoss()
  optimizer = opt.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)

  if optimizer_name == 'Adam':
    opt.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    opt.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    opt.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop

  for epoch in range(epochs):

    for batch_features, batch_labels in train_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # calculate loss
      loss = criterion(outputs, batch_labels)

      # back pass
      optimizer.zero_grad()
      loss.backward()

      # update grads
      optimizer.step()


  # evaluation
  model.eval()
  # evaluation on test data
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in test_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct/total

  return accuracy

In [25]:
!pip install optuna

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [26]:
import optuna

study = optuna.create_study(direction='maximize')

[I 2026-06-07 17:01:00,315] A new study created in memory with name: no-name-699e312b-c2a7-445f-9b39-ce30a26f49cb


In [27]:
study.optimize(objective, n_trials=10)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[I 2026-06-07 17:01:28,898] Trial 0 finished with value: 0.09166666666666666 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 128, 'epochs': 20, 'learning_rate': 0.0002923752039216547, 'dropout_rate': 0.5, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 1.0702010961327422e-05}. Best is trial 0 with value: 0.09166666666666666.
[I 2026-06-07 17:01:48,800] Trial 1 finished with value: 0.0925 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 96, 'epochs': 20, 'learning_rate': 0.00015179940728192706, 'dropout_rate': 0.30000000000000004, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 4.439374109188101e-05}. Best is trial 1 with value: 0.0925.
[I 2026-06-07 17:02:13,415] Trial 2 finished with value: 0.09583333333333334 and paramete

In [28]:
study.best_value

0.0975

In [29]:
study.best_params

{'num_hidden_layers': 2,
 'neurons_per_layer': 16,
 'epochs': 30,
 'learning_rate': 0.05659403849911812,
 'dropout_rate': 0.30000000000000004,
 'batch_size': 128,
 'optimizer': 'SGD',
 'weight_decay': 4.3698636090502645e-05}